# Compute resources

This notebook is for when a global analysis runs out of memory. It explains why the bytes a global request reads are fixed while the memory it needs is not, sizes a global two-variable request, and runs that analysis on a small cloud machine to show what it costs in practice.

It expands on Section 7 of [`subsetting-and-exporting.ipynb`](subsetting-and-exporting.ipynb), which gives the short version. The [glossary](https://github.com/carbonplan/srm-downscaling-data-utils/blob/main/GLOSSARY.md) defines terms such as *chunk*, *request* and *data read*.

**Contents:**
1. Reads versus memory
2. How much a global request reads
3. Running it on a small machine
4. Measured results

## Setup

Follow the [installation instructions](https://github.com/carbonplan/srm-downscaling-data-utils#installation) in the README, then launch JupyterLab with `pixi run jupyter lab`. The cell below imports helper functions from the repository's [`scripts/`](../scripts/README.md) folder, so open this notebook from inside the cloned repository.

In [1]:
import sys
from pathlib import Path

import flox  # noqa - xarray groupby speedup
import numpy as np
import xarray as xr

# The helper functions live in the repository's scripts/ folder.
repo = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "data_access.py").exists()), None)
if repo is None:
    raise RuntimeError("Open this notebook from inside the srm-downscaling-data-utils repository.")
sys.path.insert(0, str(repo / "scripts"))

from notebook_helpers import first_decade  # noqa: E402
from data_access import check_members, describe_request, load_downscaling_store  # noqa: E402

## 1. Reads versus memory

A global request is where a kernel most often dies partway through an analysis, because the data request is so large. The instinctive fix, a bigger machine, does not work, because memory is not the binding constraint.

**The key distinction: reads are irreducible, memory is not.**

Two variables for a decade over the global domain *must* read about 35 GB. Chunks are the unit of decompression, so every chunk overlapping your selection is fetched and decompressed in full, and no instance size changes that total. Since chunks hold roughly a year each, the read scales with the span you ask for: a single global day reads 1.6 GB, a decade about 17.5 GB per variable.

What you *can* control is how much you hold in memory at once:

| Code | What it holds in memory |
|---|---|
| `ds["tas"].sel(time=decade).load()` | All 15.2 GB of the decade at once, before anything is reduced (30.3 GB for `tas` and `pr` together). This is the usual cause of a dead kernel. |
| `ds["tas"].sel(time=decade).resample(time="YE").mean().compute()` | A few 3.8 MB chunks at a time while it works, then only the 41.5 MB of annual means it returns. On a 4 GB machine the full two-variable run peaked at 1.88 GB (Section 4). |

`chunks="auto"` is not a third option. It is a setting for opening data, not for computing: it fuses the store's 3.8 MB chunks into tasks of about 102 MB, which is 27x more memory per task for exactly the same bytes read. `load_downscaling_store(...)` already opens data with the store's own chunks, so there is nothing to change.

So a global analysis is limited by how fast the data can be read, not by how much memory the machine has. It runs on a laptop; it just takes as long as the reads take.

## 2. How much a global request reads

The cell below sizes the global request this notebook runs: `tas` and `pr` for one decade. Nothing is computed, so it returns right away.

In [2]:
# ===== CUSTOMIZE: Choose the data for the global run =====
gcm_name = "CESM2-WACCM6"  # "CESM2-WACCM6" or "UKESM1-1-LL"
method_name = "bcsd"  # "bcsd" or "qdmsd"
scenario_name = "ssp245"  # "historical", "ssp245", "g6_1p5k" or "g6_1p5k_end"
member_name = None  # None takes the pinned default
product_name = "downscaled"  # or "debiased_coarse"
global_vars = ["tas", "pr"]
# ==========================================================

# No modification needed below
# Check the pairing first: tas+pr share a member, tas+tasmax would not.
check_members(scenario_name, global_vars, gcm=gcm_name, method=method_name, product=product_name)
print()

total_read = 0
for v in global_vars:
    ds_v = load_downscaling_store(
        scenario_name, v, gcm=gcm_name, method=method_name, member=member_name,
        product=product_name,
    )
    total_read += describe_request(ds_v[v].sel(time=first_decade(ds_v)), f"Global, 10 yr, {v}")

print(f"\ncombined read for both variables: {total_read / 1e9:.1f} GB")
print("This is the floor. Reducing changes memory, not bytes read.")

CESM2-WACCM6/ssp245: tas, pr all share member 003



Global, 10 yr, tas:
  shape          {'time': 3653, 'lat': 721, 'lon': 1440}
  logical size     15.171 GB
  chunks touched     4620  (3.8 MB each, store chunks)
  data read        17.484 GB


Global, 10 yr, pr:
  shape          {'time': 3653, 'lat': 721, 'lon': 1440}
  logical size     15.171 GB
  chunks touched     4620  (3.8 MB each, store chunks)
  data read        17.484 GB

combined read for both variables: 35.0 GB
This is the floor. Reducing changes memory, not bytes read.


## 3. Running it on a small machine

The cell below reduces **before** computing, so only the annual, area-weighted result is ever held. It is off by default because it reads about 35 GB.

To run it on a genuinely small machine, use a VM in `us-west-2` — the same region as the store, so reads stay in-region instead of crossing the internet to your laptop:

```bash
pixi run -e cloud coiled run \
    --vm-type c7i.large --region us-west-2 \
    -- env RUN_GLOBAL_DEMO=1 python your_runner.py
```

`c7i.large` is 2 vCPU / **4 GB** — smaller than any machine you would reach for. Note it is deliberately *not* a `t3`: burstable instances throttle once CPU credits run out, which would make the timing measure AWS credit exhaustion rather than this technique.

For interactive work on the same size machine, `coiled notebook start --vm-type c7i.large --region us-west-2 --sync` opens JupyterLab there with your local files.

> **Careful with `--sync`.** It synchronises in *both* directions, so files that exist locally but not on the remote can be deleted from your working directory — including untracked files that git cannot restore. Commit or back up untracked work before using it. The benchmark command above deliberately omits `--sync`: a self-contained script needs nothing synced.

Measured results are recorded in Section 4.

In [3]:
# ===== CUSTOMIZE: opt in to the full global run =====
# Flip this to True, or set the RUN_GLOBAL_DEMO=1 environment variable -- which
# is how the measured numbers in Section 4 were produced on a remote VM in one command.
# Either way it reads about 35 GB, so it is off by default.
import os

RUN_GLOBAL_DEMO = os.environ.get("RUN_GLOBAL_DEMO", "0").lower() in ("1", "true", "yes")
# =====================================================

if RUN_GLOBAL_DEMO:
    import resource
    import sys
    import time

    t0 = time.perf_counter()
    reductions = {}
    for v in global_vars:
        ds_v = load_downscaling_store(
            scenario_name, v, gcm=gcm_name, method=method_name, member=member_name,
            product=product_name,
        )
        da_v = ds_v[v].sel(time=first_decade(ds_v))
        # Reduce FIRST. Annual means collapse ~3,653 daily steps into 10, and the
        # area-weighted spatial mean collapses the grid to a single number per year.
        annual_v = da_v.resample(time="YE").mean()
        weights_v = np.cos(np.deg2rad(da_v.lat))
        reductions[v] = annual_v.weighted(weights_v).mean(dim=["lat", "lon"])

    # One graph for both variables, so each is streamed through once.
    result = xr.Dataset(reductions).compute()

    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    peak_gb = peak / 1024**3 if sys.platform == "darwin" else peak / 1024**2

    print(result)
    print(f"\npeak RSS   {peak_gb:6.2f} GB")
    print(f"wall time  {time.perf_counter() - t0:6.0f} s")
    print(f"result     {result.nbytes} bytes held, from {total_read / 1e9:.1f} GB read")
else:
    print("RUN_GLOBAL_DEMO is False - skipping the ~35 GB global run.")
    print("Measured results from a 4 GB VM are recorded in Section 4.")


RUN_GLOBAL_DEMO is False - skipping the ~35 GB global run.
Measured results from a 4 GB VM are recorded in Section 4.


## 4. Measured results

Recorded from a single `c7i.large` (2 vCPU, **4 GB RAM**) in `us-west-2`, running the cell above with `RUN_GLOBAL_DEMO=1`:

| Metric | Value |
|---|---|
| Variables | `tas`, `pr` (CESM2-WACCM6, bcsd, ssp245, member `003`) |
| Window | 2015-2024, global (`time=3653, lat=721, lon=1440`) |
| Logical size | 30.3 GB |
| Data read | 35.0 GB |
| **Peak RSS** | **1.88 GB** |
| Wall time | 829 s (~14 min) |
| Result held | 160 bytes |

**35 GB streamed through a machine with 4 GB of RAM, and peak memory stayed at 1.88 GB.** A machine with 192 GB — nearly fifty times this one — will still die on the same workload if it calls `.load()`. Memory was never the constraint; holding the data was.

That is the entire technique: nothing larger than a handful of 3.8 MB chunks is ever resident, so peak memory is set by chunk size and task concurrency, not by how much data you traverse.

The honest cost is **time**: about 14 minutes at roughly 42 MB/s end to end, bounded by decompression on 2 vCPUs rather than by RAM. More cores would shorten it; more memory would not. That is the axis worth scaling.

**If you must keep gridded output** (a map rather than a time series), reduce along time first — `resample(time="YE").mean()` on a global decade produces 41.5 MB, which writes to Zarr comfortably. Calling `.load()` on the unreduced 30.3 GB does not.